In [0]:
# CELL: Install BeautifulSoup
#-----------------------------------------------------------

# 1. Install BeautifulSoup
%pip install beautifulsoup4

# 2. Verify Installation
try:
    from bs4 import BeautifulSoup
    print("BeautifulSoup is installed and working correctly.")
except ImportError:
    raise ImportError(
        "BeautifulSoup failed to install.  ",  
        "Please rerun this cell"
    )

In [0]:
# CELL: Install Python-Dotenv
#-----------------------------------------------------------

# 1. Install Python-Dotenv
%pip install python-dotenv

# 2. Verify Installation
try:
    from dotenv import load_dotenv
    print("Dotenv is installed and working correctly.")
except ImportError:
    raise ImportError("Dotenv failed to install.  Please rerun this cell")

In [0]:
# CELL: Import Required Libraries, Read Environment Variables

import requests
from bs4 import BeautifulSoup
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv()

# 2. Assign storage acount name and key vars from env
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
storage_account_key = os.getenv("STORAGE_ACCOUNT_KEY")

# 3. Verify env read successfully
print(f"storage_account_name = {storage_account_name}")

In [0]:
# 1. Assign base_page_url
base_page_url = "https://www.dubaipulse.gov.ae/"

# 1. Assign data source URL
txns_page_url = f"{base_page_url}dld-transactions/dld_transactions-open"

# 2. Assign transactions landing destination
landing_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/landing/"

# 3. Assign transactions archival destination
archive_zone = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/transactions/archive/"

# 4. Assign requests headers
requests_headers = {
    "User-Agent": "Mozilla/5.0"
}

In [0]:
# AUTO-CREATE-FIX: Configuration auto-creates the 'bronze' container if it is missing
spark.conf.set(
    f"fs.azure.createRemoteFileSystemDuringInitialization", "true"
)

# Connect to storage account
spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key,
)

print("Configuration loaded & Auto-create enabled")

In [0]:
def get_dynamic_download_url(page_url: str):
    """
    Scrapes the provided URL for a dynamic download link and returns it.
    If no dynamic download link is found, raises a ValueError.
    Args:
        page_url (str): The URL to scrape for the dynamic download link.
    Returns:
        str: The dynamic download link.
    Raises:
        ValueError: If no dynamic download link is found on the page.
    """
    print(f"Scraping {page_url}...")

    response = requests.get(page_url, headers=requests_headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.content, "html.parser")

    links = soup.find_all("a", href=True)
    for link in links:
        href = link["href"]
        if "download" in href and "transactions.csv" in href:
            final_url = f"{base_page_url}{href}" if href.startswith("/") else href
            print(f"found dynamic link: {final_url}")
            return final_url

    raise ValueError("Could not find the download link on the page.")


In [0]:
def cleanup_old_files(archive_zone: str, days_to_keep: int = 14):
    """
    Deletes files in the archive zone that are older than the specified number of days.
    Args:
        archive_zone (str): The path to the archive zone.
        days_to_keep (int, optional): The number of days to keep files in the archive zone. Defaults to 14.
    Returns:
        None
    """
    print("Checking archive for old files...")
    cutoff_date = datetime.now() - timedelta(days=days_to_keep)

    try:
        files = dbutils.fs.ls(archive_zone)
        for file in files:
            try:
                filename = file.name.replace(".csv", "")
                file_date = datetime.strptime(filename, "%Y-%m-%d")
                if file_date < cutoff_date:
                    print(f"Removing file: {file.path}")
                    dbutils.fs.rm(file.path)
            except ValueError:
                continue
    except Exception as e:
        print(f"Archive is empty or does not exist. Received Error: {str(e)}")

cleanup_old_files(archive_zone=archive_zone)

In [0]:
def archive_current_file(landing_zone_dir: str, archive_zone_dir: str):
    """
    Archives the current file in the landing zone to the archive zone.
    Args:
        landing_zone_dir (str): The path to the landing zone directory.
        archive_zone_dir (str): The path to the archive zone directory
    Returns:
        None
    """
    current_file_path = f"{landing_zone_dir}current.csv"

    try:
        dbutils.fs.ls(current_file_path)

        today_str = datetime.now().strftime("%Y-%m-%d")
        archive_path = f"{archive_zone_dir}{today_str}.csv"

        print(f"Archiving previous 'current.csv' to '{archive_path}'")
        dbutils.fs.mv(current_file_path, archive_path)
    except Exception as e:
        print(f"No 'current.csv' found in '{landing_zone_dir}'. Skipping archiving")

archive_current_file(landing_zone_dir=landing_zone, archive_zone_dir=archive_zone)

In [0]:
import os
import time

def download_snapshot_with_retry(page_url: str, dest_path: str, max_retries:int = 3):
    """
    Downloads a snapshot from the provided URL and saves it to the landing zone.
    If the download fails, it retries up to a maximum number of retries.
    Args:
        page_url (str): The URL from which to download.
        dest_path (str): The path to save the downloaded snapshot
        max_retries (int, optional): The maximum number of retries. Defaults to 3.
    Returns:
        None
    """
    TEN_MBYTES = 10 * 1024 * 1024
    REQUESTS_TIMEOUT_SECS = 30
    LOCAL_PATH = "/tmp/current.csv"

    fresh_url = get_dynamic_download_url(page_url)

    for attempt in range(1, max_retries + 1):
        try:
            print(f"Attempt {attempt}/{max_retries}: Downloading to driver node...")
            with requests.get(fresh_url, headers=requests_headers, stream=True, timeout=REQUESTS_TIMEOUT_SECS) as rsp:
                rsp.raise_for_status()
                with open(LOCAL_PATH, "wb") as f:
                    for chunk in rsp.iter_content(chunk_size=TEN_MBYTES):
                        if chunk:
                            f.write(chunk)
                            print("Download complete.")
                            break
        except Exception as e:
            print(f"Attempt {attempt} failed: {str(e)}")
            if attempt < max_retries:
                time.sleep(5) # sleep and retry
            else:
                print("All retries failed...")
                raise e

    if os.path.exists(LOCAL_PATH):
        print(f"Uploading to Bronze Landing Zone...")
        dbutils.fs.cp(f"file:{LOCAL_PATH}", f"{dest_path}current.csv")
        os.remove(LOCAL_PATH)

        print(f"Success! 'current.csv' is updated in the Data Lake.")
    
download_snapshot_with_retry(page_url=txns_page_url, dest_path=landing_zone)